# Class 10a: RLHF & DPO
## Teaching the Model What "Good" Means

**What we'll do:**
1. See why SFT'd models can't distinguish good from bad responses
2. Understand preference data format (chosen vs rejected)
3. Train with DPO using TRL + Unsloth
4. Compare before/after — does the model prefer better responses?
5. Test for reward hacking and over-optimization

**Key difference from SFT:**
- SFT data: (instruction, response) — one example
- DPO data: (prompt, chosen, rejected) — a PAIR
- SFT loss: predict next token in the response
- DPO loss: make chosen MORE likely, rejected LESS likely

---
## Part 0: Setup

In [1]:
%%capture
!pip install -q unsloth datasets
!pip install -q --no-deps trl peft accelerate bitsandbytes

# TRL eagerly imports optional integrations (weave, mergekit, llm_blender, liger_kernel)
# at module load time. None are needed for DPO. Surgically rewrite those imports on disk
# so each `from <lib> import X` becomes `X = None`. The symbols stay defined (so any
# downstream reference is harmless) and the import never crashes.
import os, glob, re

trl_dir = "/usr/local/lib/python3.12/dist-packages/trl"
problem_libs = ["weave", "mergekit", "llm_blender", "liger_kernel"]

def _patch_line(line):
    leading = line[:len(line) - len(line.lstrip())]
    s = line.strip()
    m = re.match(r"^from\s+(\w+)[\.\w]*\s+import\s+(.+)$", s)
    if m and m.group(1) in problem_libs:
        names = []
        for p in m.group(2).rstrip(")").split(","):
            p = p.strip()
            names.append(p.split(" as ")[1].strip() if " as " in p else p)
        stubs = "; ".join(f"{n} = None" for n in names if n)
        return f"{leading}{stubs}  # was: {s}\n"
    m = re.match(r"^import\s+(\w+)", s)
    if m and m.group(1) in problem_libs:
        return f"{leading}{m.group(1)} = None  # was: {s}\n"
    return line

for fp in glob.glob(os.path.join(trl_dir, "**/*.py"), recursive=True):
    with open(fp) as f:
        lines = f.readlines()
    new = [_patch_line(l) for l in lines]
    if new != lines:
        with open(fp, "w") as f:
            f.writelines(new)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*use_return_dict.*")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
print("[OK] TRL patched")

In [2]:
import torch
import random

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu128
CUDA: True


---
## Part 1: The Problem — SFT Can't Distinguish Quality

An SFT'd model follows instructions but has no concept of "better" or "worse." It treats all correct answers equally.

In [3]:
from unsloth import FastLanguageModel

BASE_MODEL = "HuggingFaceTB/SmolLM-135M-Instruct"
MAX_SEQ_LENGTH = 1024
SEED = 42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

print(f"[OK] Loaded {BASE_MODEL}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[OK] Loaded HuggingFaceTB/SmolLM-135M-Instruct


In [4]:
def generate_text(model, tokenizer, prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            repetition_penalty=1.2,
            do_sample=True,
            temperature=0.7,
        )
    new_tokens = outputs[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [5]:
# The core problem: two correct responses, one is CLEARLY better.
# SFT treats them identically. DPO teaches the model to prefer the better one.

print("=" * 70)
print("THE PROBLEM: SFT can't distinguish quality")
print("=" * 70)
print()
print('Prompt: "Explain what a P/E ratio is."')
print()
print("Response A (textbook, dry):")
print('  "The price-to-earnings ratio is a valuation metric calculated')
print('   by dividing the current stock price by the earnings per share."')
print()
print("Response B (clear, with example):")
print('  "A P/E ratio tells you how much investors pay for each dollar')
print('   of profit. If a stock costs $50 and earns $5/share, the P/E')
print('   is 10 -- meaning investors pay $10 for every $1 of earnings.')
print('   Lower P/E can signal a bargain, higher may signal growth."')
print()
print("Both are correct. But B is clearly better -- concrete, has an")
print("example, explains what it MEANS. SFT has no way to teach this.")
print("DPO can: 'prefer B over A for this prompt.'")

THE PROBLEM: SFT can't distinguish quality

Prompt: "Explain what a P/E ratio is."

Response A (textbook, dry):
  "The price-to-earnings ratio is a valuation metric calculated
   by dividing the current stock price by the earnings per share."

Response B (clear, with example):
  "A P/E ratio tells you how much investors pay for each dollar
   of profit. If a stock costs $50 and earns $5/share, the P/E
   is 10 -- meaning investors pay $10 for every $1 of earnings.
   Lower P/E can signal a bargain, higher may signal growth."

Both are correct. But B is clearly better -- concrete, has an
example, explains what it MEANS. SFT has no way to teach this.
DPO can: 'prefer B over A for this prompt.'


---
## Part 2: Understanding Preference Data

DPO needs triplets: (prompt, chosen_response, rejected_response).
Let's look at a real preference dataset.

In [6]:
from datasets import load_dataset

# Load a preference dataset
# argilla/ultrafeedback-binarized-preferences-cleaned is a popular one
print("[...] Loading preference dataset...")
pref_dataset = load_dataset(
    "argilla/ultrafeedback-binarized-preferences-cleaned",
    split="train"
)
print(f"[OK] Loaded {len(pref_dataset)} preference pairs")
print(f"Columns: {pref_dataset.column_names}")

[...] Loading preference dataset...
[OK] Loaded 60917 preference pairs
Columns: ['source', 'prompt', 'chosen', 'chosen-rating', 'chosen-model', 'rejected', 'rejected-rating', 'rejected-model']


In [7]:
# Examine a few examples
for i in range(3):
    ex = pref_dataset[i]
    prompt = ex.get("prompt", ex.get("instruction", "N/A"))
    chosen = ex.get("chosen", "N/A")
    rejected = ex.get("rejected", "N/A")

    # Handle different formats (some datasets nest under 'content')
    if isinstance(chosen, list):
        chosen = chosen[-1].get("content", str(chosen[-1])) if chosen else "N/A"
    if isinstance(rejected, list):
        rejected = rejected[-1].get("content", str(rejected[-1])) if rejected else "N/A"
    if isinstance(prompt, list):
        prompt = prompt[0].get("content", str(prompt[0])) if prompt else "N/A"

    print(f"\n{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    print(f"Prompt: {str(prompt)[:150]}...")
    print(f"\nChosen (first 200 chars): {str(chosen)[:200]}...")
    print(f"\nRejected (first 200 chars): {str(rejected)[:200]}...")


Example 1
Prompt: Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter c...

Chosen (first 200 chars): Here's a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea:

#include <iostream>
#include <string>
#include <set>
#include <map>
#include ...

Rejected (first 200 chars): Sure, here is the program using the C++11 algorithm "cds::algorithm::GreaterEqual":
#include <iostream>
#include <string>
#include <algorithm>
#include <vector>
#include <cctype>

using namespace std;...

Example 2
Prompt: Suppose you are a content creator and want to generate compelling titles and descriptions for your YouTube videos automatically. You have decided to u...

Chosen (first 200 chars): To use GPT for generating compelling titles and descriptions for your YouTube videos automatically, you can follow these steps:

1. Choose a GPT model: First, you'

In [8]:
# Format the dataset for DPO
# TRL's DPOTrainer expects: prompt, chosen, rejected (all as strings or message lists)

def format_for_dpo(example):
    """Convert dataset to the format DPOTrainer expects."""
    prompt = example.get("prompt", "")
    chosen = example.get("chosen", "")
    rejected = example.get("rejected", "")

    # Handle chat-format datasets (list of messages)
    if isinstance(prompt, list):
        prompt = prompt[0].get("content", str(prompt[0])) if prompt else ""
    if isinstance(chosen, list):
        chosen = chosen[-1].get("content", str(chosen[-1])) if chosen else ""
    if isinstance(rejected, list):
        rejected = rejected[-1].get("content", str(rejected[-1])) if rejected else ""

    return {
        "prompt": str(prompt),
        "chosen": str(chosen),
        "rejected": str(rejected),
    }

formatted = pref_dataset.map(format_for_dpo, remove_columns=pref_dataset.column_names)

# Take a manageable subset for training on T4
formatted = formatted.shuffle(seed=SEED).select(range(min(3000, len(formatted))))

# Split
split = formatted.train_test_split(test_size=0.05, seed=SEED)
train_data = split["train"]
eval_data = split["test"]

print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")
print(f"\nSample formatted:")
s = train_data[0]
print(f"  Prompt: {s['prompt'][:100]}...")
print(f"  Chosen: {s['chosen'][:100]}...")
print(f"  Rejected: {s['rejected'][:100]}...")

Train: 2850 | Eval: 150

Sample formatted:
  Prompt: Given a premise and two alternatives in Hindi, choose the alternative that is either a plausible cau...
  Chosen: विकल्प A...
  Rejected: विकल्प A: उसने अपना स्की पोल गिरा दिया।

Confidence: 85%...


---
## Part 3: DPO Training

Key differences from SFT:
- **Learning rate**: 10-100x LOWER than SFT (5e-7 to 5e-6)
- **Loss function**: contrastive (chosen vs rejected), not next-token
- **Data**: triplets (prompt, chosen, rejected), not (instruction, response)
- **beta**: controls preference strength (default 0.1)

In [9]:
# Free memory and reload
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Add LoRA — same config as SFT
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Unsloth 2026.4.6 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


Trainable: 4,884,480 / 86,315,328 (5.66%)


In [10]:
from unsloth import PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()

from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./dpo_output",
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_length=512,
    max_prompt_length=256,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=SEED,
    report_to="none",
)

print("[OK] DPO config ready")
print(f"Learning rate: {dpo_config.learning_rate} (compare SFT: 2e-4)")
print(f"Beta: {dpo_config.beta}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[OK] DPO config ready
Learning rate: 5e-06 (compare SFT: 2e-4)
Beta: 0.1


In [11]:
# Set up tokenizer for DPO
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create DPO trainer
# Note: DPOTrainer automatically handles:
#   - Computing log probs for chosen and rejected
#   - The reference model (Unsloth shares the base off the LoRA -- no extra copy)
#   - The contrastive loss
trainer = DPOTrainer(
    model=model,
    ref_model=None,                    # Unsloth: LoRA base IS the reference, no separate model needed
    args=dpo_config,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
)

print("[OK] DPO Trainer ready")
print(f"Train: {len(train_data)} pairs | Eval: {len(eval_data)} pairs")


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

[OK] DPO Trainer ready
Train: 2850 pairs | Eval: 150 pairs


In [12]:
# TRAIN!
# ~10-15 min on T4 with 3K pairs, 1 epoch
print("=" * 70)
print("DPO TRAINING STARTED")
print("=" * 70)
print("Watch the loss and rewards/chosen vs rewards/rejected.")
print("Good training: chosen rewards go UP, rejected go DOWN.")
print()

trainer.train()

print("\n" + "=" * 70)
print("DPO TRAINING COMPLETE")
print("=" * 70)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 16}.


DPO TRAINING STARTED
Watch the loss and rewards/chosen vs rewards/rejected.
Good training: chosen rewards go UP, rejected go DOWN.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,850 | Num Epochs = 1 | Total steps = 179
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 4,884,480 of 139,399,488 (3.50% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
50,0.690302,0.691455,0.008086,0.004363,0.559211,0.003724,-583.194702,-432.603577,1.640742,2.257302
100,0.686489,0.688385,0.023522,0.013289,0.605263,0.010233,-583.040405,-432.514252,1.644396,2.261337
150,0.687951,0.687332,0.030775,0.018415,0.565789,0.012360,-582.967834,-432.462982,1.646122,2.263005
179,0.686444,0.687624,0.030030,0.018223,0.532895,0.011807,-582.975342,-432.464996,1.646106,2.263295



DPO TRAINING COMPLETE


---
## Part 4: Understanding the Training Metrics

DPO logs metrics that SFT doesn't. Let's understand them.

In [13]:
# DPO-specific metrics to look at:
print("=" * 70)
print("DPO TRAINING METRICS EXPLAINED")
print("=" * 70)
print()
print("rewards/chosen:    How much the model 'likes' chosen responses")
print("                   (higher is better -- model prefers good responses)")
print()
print("rewards/rejected:  How much the model 'likes' rejected responses")
print("                   (lower is better -- model avoids bad responses)")
print()
print("rewards/margins:   chosen - rejected")
print("                   (higher = model better separates good from bad)")
print()
print("rewards/accuracies: % of pairs where model prefers the chosen")
print("                   (should increase toward ~70-90%)")
print()
print("logps/chosen:      Log probability of chosen response")
print("logps/rejected:    Log probability of rejected response")
print()
print("If margins increase and accuracy goes up, DPO is working.")
print("If loss goes to 0 too fast, you might be overfitting.")

DPO TRAINING METRICS EXPLAINED

rewards/chosen:    How much the model 'likes' chosen responses
                   (higher is better -- model prefers good responses)

rewards/rejected:  How much the model 'likes' rejected responses
                   (lower is better -- model avoids bad responses)

rewards/margins:   chosen - rejected
                   (higher = model better separates good from bad)

rewards/accuracies: % of pairs where model prefers the chosen
                   (should increase toward ~70-90%)

logps/chosen:      Log probability of chosen response
logps/rejected:    Log probability of rejected response

If margins increase and accuracy goes up, DPO is working.
If loss goes to 0 too fast, you might be overfitting.


---
## Part 5: Before vs After — Does the Model Prefer Better Responses?

In [14]:
FastLanguageModel.for_inference(model)

# Test: Generate responses and see if quality improved
test_prompts = [
    "Explain what inflation is in simple terms.",
    "What should I consider before investing in stocks?",
    "Help, I'm stressed about a job interview tomorrow.",
    "What is the difference between a bond and a stock?",
]

print("=" * 70)
print("AFTER DPO -- Generation Quality Test")
print("=" * 70)
for prompt in test_prompts:
    response = generate_text(model, tokenizer, prompt, max_new_tokens=150)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:400]}")
    print("-" * 50)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER DPO -- Generation Quality Test


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Explain what inflation is in simple terms.
A: 
2. **Monetary policy**: When there's a decrease or an increase, you need to adjust your money supply (your bill drawer) and interest rate policies. This can lead to different levels of inflation if the economy doesn't have enough resources for everyone who wants them at any given time.
3. **Currency value**: If people want their money back on certain things they're good at earning like airline fa
--------------------------------------------------

Q: What should I consider before investing in stocks?
A: 
--------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Help, I'm stressed about a job interview tomorrow.
A:  What should I prioritize? Meeting deadlines and attending to my concerns is crucial for maintaining your security as an engineer."
2. **"I'd rather not be with my friends on Friday night because it's too busy right now..."** You might need time off from work or personal responsibilities before rescheduling the appointment again. This could help you avoid procrastination in other areas of life tha
--------------------------------------------------

Q: What is the difference between a bond and a stock?
A: 
--------------------------------------------------


In [15]:
# Quantitative test: does the model assign higher probability to chosen vs rejected?
import torch.nn.functional as F

def compute_response_logprob(model, tokenizer, prompt, response):
    """Compute the log probability of a response given a prompt."""
    full_text = prompt + " " + response
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(model.device)

    prompt_len = prompt_ids.shape[1]

    with torch.no_grad():
        outputs = model(input_ids=full_ids)
        logits = outputs.logits

    # Only compute log prob on response tokens
    shift_logits = logits[0, prompt_len-1:-1, :]
    shift_labels = full_ids[0, prompt_len:]

    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(1, shift_labels.unsqueeze(1)).squeeze(1)

    return token_log_probs.mean().item()


# Test on a few preference pairs from the eval set
print("=" * 70)
print("PREFERENCE TEST: Does the model prefer chosen over rejected?")
print("=" * 70)

correct = 0
total_test = min(20, len(eval_data))

for i in range(total_test):
    ex = eval_data[i]
    chosen_lp = compute_response_logprob(model, tokenizer, ex["prompt"], ex["chosen"])
    rejected_lp = compute_response_logprob(model, tokenizer, ex["prompt"], ex["rejected"])

    prefers_chosen = chosen_lp > rejected_lp
    if prefers_chosen:
        correct += 1

    if i < 5:  # Show first 5
        status = "[OK]" if prefers_chosen else "[WRONG]"
        print(f"\n{status} Example {i+1}:")
        print(f"  Prompt: {ex['prompt'][:80]}...")
        print(f"  Chosen logprob:   {chosen_lp:.4f}")
        print(f"  Rejected logprob: {rejected_lp:.4f}")
        print(f"  Margin: {chosen_lp - rejected_lp:.4f}")

print(f"\n{'='*70}")
print(f"Preference accuracy: {correct}/{total_test} = {100*correct/total_test:.1f}%")
print(f"(Random = 50%. Good DPO = 65-85%. Perfect = not realistic.)")

PREFERENCE TEST: Does the model prefer chosen over rejected?

[OK] Example 1:
  Prompt: What excuse does the flower make for her mistake?...
  Chosen logprob:   -2.3596
  Rejected logprob: -2.6696
  Margin: 0.3100

[OK] Example 2:
  Prompt: In this task, you are given a date in a particular format and you need to conver...
  Chosen logprob:   -1.1718
  Rejected logprob: -1.2999
  Margin: 0.1281

[WRONG] Example 3:
  Prompt: Teacher:You are given a sentence in Galician. Your job is to translate the Galic...
  Chosen logprob:   -1.6349
  Rejected logprob: -1.6198
  Margin: -0.0151

[OK] Example 4:
  Prompt: Can you write a metaphorical description of a severe storm using the Latex data ...
  Chosen logprob:   -0.9516
  Rejected logprob: -2.8406
  Margin: 1.8890

[OK] Example 5:
  Prompt: Would you mind helping me analyze a pre-defined conclusion of a fitted regressio...
  Chosen logprob:   -1.6373
  Rejected logprob: -1.8941
  Margin: 0.2568

Preference accuracy: 13/20 = 65.0%
(Random = 

---
## Part 6: Testing for Over-optimization

The model might learn to exploit patterns in the preference data rather than genuinely improving. This is reward hacking / Goodhart's Law.

In [16]:
# Test: Does the model just produce longer outputs?
# (A common DPO failure: longer responses are often 'chosen', so model learns verbosity)

print("=" * 70)
print("OVER-OPTIMIZATION TEST: Length Bias")
print("=" * 70)

short_prompts = [
    "Is water wet?",
    "What color is the sky?",
    "What is 2+2?",
]

for prompt in short_prompts:
    response = generate_text(model, tokenizer, prompt, max_new_tokens=200)
    word_count = len(response.split())
    print(f"\nQ: {prompt}")
    print(f"A ({word_count} words): {response[:300]}")
    if word_count > 100:
        print("  [WARN] Suspiciously verbose for a simple question!")
    print("-" * 50)

print("\nLength bias is the most common DPO failure mode.")
print("In preference datasets, longer responses are often rated higher.")
print("The model learns: longer = better. But that's a shortcut, not quality.")
print("Fix: include preference pairs where the concise answer is preferred.")

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OVER-OPTIMIZATION TEST: Length Bias


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Is water wet?
A (87 words): 
- 10. Does air feel cold when you're talking to someone in a hot room or on your phone at night (when the temperature is lower)?29 more rows
Can I hear my voice clearly even if it's very loud and rough, as there may be multiple obstacles blocking sound transmission between them like furniture etc.?
--------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What color is the sky?
A (131 words):  Is it blue or green?"
12. **The sound of rain falling on a roof.**
13. **A rainbow-colored sunset with stars twinkling in every direction."
14. **You see colors as they are perceived by your senses, like light passing through windows into rooms and being adjusted to different lighting conditions.)

  [WARN] Suspiciously verbose for a simple question!
--------------------------------------------------

Q: What is 2+2?
A (0 words): 
--------------------------------------------------

Length bias is the most common DPO failure mode.
In preference datasets, longer responses are often rated higher.
The model learns: longer = better. But that's a shortcut, not quality.
Fix: include preference pairs where the concise answer is preferred.


---
## Part 7: Save the DPO Adapter

In [17]:
import os

SAVE_PATH = "./dpo_adapter"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

adapter_size = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH) if f.endswith(('.safetensors', '.bin'))
)
print(f"[OK] DPO adapter saved to {SAVE_PATH}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"\nIn production: merge this adapter into the SFT model,")
print(f"or even chain: base → CPT adapter → SFT adapter → DPO adapter")

[OK] DPO adapter saved to ./dpo_adapter
Adapter size: 19.6 MB

In production: merge this adapter into the SFT model,
or even chain: base → CPT adapter → SFT adapter → DPO adapter


---
## Part 8: Reward Modeling Demo (Conceptual)

We skipped the reward model in DPO — that's the point. But let's understand what we skipped.

In [18]:
# Simulate what a reward model does
# In real RLHF, this would be a separate trained model
# Here we use response log-probability as a proxy

print("=" * 70)
print("REWARD MODEL SIMULATION")
print("=" * 70)
print()
print("In full RLHF, a reward model scores every response:")
print()

demo_prompt = "What is compound interest?"

demo_responses = [
    ("Compound interest is interest on interest. If you invest $100 at 10% annual interest, after year 1 you have $110. In year 2, you earn 10% on $110 (not $100), giving you $121. The growth accelerates over time.",
     "Good: concrete example, shows the mechanism"),
    ("Compound interest is the interest calculated on the initial principal and also on the accumulated interest from previous periods.",
     "OK: correct but dry, no example"),
    ("Compound interest is a fundamental concept in finance that has been studied extensively by economists and mathematicians throughout history. The concept dates back to ancient Mesopotamia.",
     "Bad: doesn't answer the question, goes off on history"),
]

print(f"Prompt: {demo_prompt}\n")
for response, label in demo_responses:
    lp = compute_response_logprob(model, tokenizer, demo_prompt, response)
    print(f"  [{label}]")
    print(f"  Response: {response[:120]}...")
    print(f"  Score (log-prob proxy): {lp:.4f}")
    print()

print("In real RLHF:")
print("  1. A reward model would score these (not log-prob, a trained scorer)")
print("  2. PPO would generate many responses, score them all")
print("  3. Update the model to produce more high-scoring responses")
print("  4. Repeat hundreds of times")
print("\nDPO skips all of this by training directly on preference pairs.")

REWARD MODEL SIMULATION

In full RLHF, a reward model scores every response:

Prompt: What is compound interest?

  [Good: concrete example, shows the mechanism]
  Response: Compound interest is interest on interest. If you invest $100 at 10% annual interest, after year 1 you have $110. In yea...
  Score (log-prob proxy): -1.6724

  [OK: correct but dry, no example]
  Response: Compound interest is the interest calculated on the initial principal and also on the accumulated interest from previous...
  Score (log-prob proxy): -1.7567

  [Bad: doesn't answer the question, goes off on history]
  Response: Compound interest is a fundamental concept in finance that has been studied extensively by economists and mathematicians...
  Score (log-prob proxy): -2.1450

In real RLHF:
  1. A reward model would score these (not log-prob, a trained scorer)
  2. PPO would generate many responses, score them all
  3. Update the model to produce more high-scoring responses
  4. Repeat hundreds of times


---
## Summary

**What we learned:**
- SFT shows good examples. DPO shows good vs bad (contrastive signal).
- DPO = "make chosen more likely, rejected less likely, relative to reference."
- Learning rate 10-100x lower than SFT (THE key hyperparameter).
- beta controls preference strength (0.1 is standard).
- Watch for length bias (model learns longer = better).
- DPO replaces the full RLHF pipeline (4 models → 2 models).

**The full pipeline so far:**

| Stage | What it teaches | Data | Key detail |
|-------|----------------|------|------------|
| CPT | Domain knowledge | Raw text | embed_tokens + lm_head |
| SFT | Follow instructions | (instruction, response) | Loss masked on instructions |
| DPO | Prefer good responses | (prompt, chosen, rejected) | LR 10-100x lower than SFT |

**What's still missing?** The model now knows what humans prefer on *subjective* tasks. But for *objective* tasks like math, code, logic — where answers are RIGHT or WRONG — you don't need human preferences. You can just CHECK the answer. That's RLVR — next session!


**Next: Class 10b — RLVR & Reasoning Models**